## Offline Reader Mode: Constructed Descriptive Financial Analysis

The current code uses only a Seed 101 constructed panel of 180 months and skips French observations. The historical report, figure, and exercise answers are separate reading materials. French raw, extracted, and processed data are not bundled, and the rights and permitted uses have not been established; this Notebook does not analyze those data.

Run from the package root as described in the [reader guide](../READER_GUIDE.md). 

# Project 1: Financial Data Analysis

Identify the data object, monthly intervals and decimal units before calculating wealth, risk and fixed segments for the current constructed panel. NoDur, HiTec and Enrgy are method-example labels here, not French observations. The constructed main table uses 120 months over 2010–2019; three 60-month segments illustrate window sensitivity, not market out-of-sample testing.

The [historical report](../research_report/report.md) uses the October 5, 2026 French snapshot. Within the [six exercises](../exercises.md) and [complete solutions](../solutions.md), only report-reading Exercises 2 and 5 use that snapshot; the other questions use their own hand inputs or interpret figure axes.

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("project01_financial_data_analysis_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
PROJECT = ROOT / "projects/project01"
sys.path.insert(0, str(PROJECT))
from project01_analysis import load_config, make_project_figure, summarize_all_windows
from quant_course.performance import drawdown_path, wealth_index


## 1. Constructed Inputs and Historical Snapshots

The next code cell reads the shared configuration, generates three constructed simple-return columns for 180 months with Seed101, fixes RF at 0.001 and writes constructed_monthly_returns.csv under the printed RUN_DIR. It does not call load_project_data to load French observations.

The historical working-course route instead checks manifests and ZIP/extracted/processed CSV hashes and aligns French returns with RF. The [dictionary](../data_dictionary.md) retains that account. A hash establishes neither historical availability nor permission to execute that route here.

In [ ]:
config = load_config(PROJECT / "configs/analysis.json")
# CONSTRUCTED: synthetic monthly returns; labels are method examples, not French observations.
rng = np.random.default_rng(101)
index = pd.period_range("2010-01", "2024-12", freq="M", name="month")
data = pd.DataFrame(rng.normal(0.004, 0.035, (len(index), 3)), index=index,
                    columns=[x["column"] for x in config["industries"]])
data["RF"] = 0.001
print("CONSTRUCTED panel:", len(data), "continuous months; French branch SKIPPED: rights unknown")
assert not data.isna().any().any()
data.to_csv(RUN_DIR / "constructed_monthly_returns.csv")


## 2. Time and Units in the Current Main Window

month labels return intervals in a continuous monthly PeriodIndex, not releases or trades. Constructed values already use decimals: 0.01 means 1%; do not divide by 100 again. RF=0.001 is aligned by month, separate from annual rates and observed Treasury-bill returns.

The next cell checks the complete January 2010–December 2019 index and 120 months, then displays the constructed main table and RF mean. The unchanged display labels “主窗口” and “RF 同窗口月均（百分比 / 月）” mean “main window” and “mean RF in the same window, percent per month.” Historical release, ingestion, signal and trade clocks are absent.

In [ ]:
main = data.loc["2010-01":"2019-12"].copy()
expected_main = pd.period_range("2010-01", "2019-12", freq="M", name="month")
assert main.index.equals(expected_main) and len(main) == 120

display(main.head())
print("主窗口：", main.index[0], "到", main.index[-1], "；", len(main), "个连续月度区间")
print("RF 同窗口月均（百分比 / 月）:", f"{main['RF'].mean() * 100:.4f}%")

## 3. Simple Returns, Log Returns and Wealth

The next cell is a separate hand calculation: initial wealth 100, +50%, then −50%. It uses neither the Seed101 main table nor French data. Without external flows, W_t=W_{t-1}(1+R_t), yielding 100, 150, 75; drawdown uses the running peak.

For R_t>−1, g_t=ln(1+R_t), and across time Σg_t=ln(W_T/W_0). Log returns add over time but cannot replace capital-weighted simple returns across assets. Preserved output labels mean: 构造收益 = constructed returns, 财富路径 = wealth path, 路径回撤 = path drawdown, 算术平均 = arithmetic mean, 平均对数收益 = mean log return.

In [ ]:
# 构造的两期例子只用于手算复利，不作为市场观测。
toy_returns = np.array([0.50, -0.50])
toy_wealth = wealth_index(toy_returns, initial_capital=100.0)
toy_drawdown = drawdown_path(toy_wealth)
print("构造收益:", toy_returns)
print("财富路径:", toy_wealth)
print("路径回撤:", toy_drawdown)
print("算术平均:", toy_returns.mean())
print("平均对数收益:", np.log1p(toy_returns).mean())
np.testing.assert_allclose(toy_wealth, [100.0, 150.0, 75.0])
np.testing.assert_allclose(toy_drawdown, [0.0, 0.0, -0.5])

Next compare monthly simple-return and log-return means for the current Seed101 constructed main window. pct_per_month denotes displayed percentages; inputs remain decimals. The arithmetic mean describes a single-period average; the log mean describes this path’s average log growth. wealth_index still consumes simple returns; initial wealth 1 is not a price. The unchanged industry labels are 非耐用品 = Nondurables (NoDur), 高科技 = High Technology (HiTec), 能源 = Energy (Enrgy).

In [ ]:
mean_returns = pd.DataFrame({
    "industry": [item["label_zh"] for item in config["industries"]],
    "mean_simple_return_pct_per_month": [main[item["column"]].mean() * 100 for item in config["industries"]],
    "mean_log_return_pct_per_month": [np.log1p(main[item["column"]]).mean() * 100 for item in config["industries"]],
})
display(mean_returns.round(4))

## 4. Risk and Growth in the Constructed Panel

Volatility is the monthly simple-return sample standard deviation (ddof=1) multiplied by √12: a conventional annual scale, not an identity for annual compound-return risk. Same-month e_t=R_t−RF_t enters Sharpe as √12 times excess-return mean divided by its sample standard deviation, which must be positive. This is neither an Alpha nor a significance test.

Drawdown is W_t/running peak−1, with maximum loss displayed positively and initial wealth retained. CAGR=(W_end/W_start)^(12/n_months)−1 uses the complete-month span without external flows. The next main table uses constructed inputs only. Its preserved headers mean: 行业 = industry; 期末财富指数 = final wealth index; 累计收益小数 = cumulative decimal return; CAGR 小数 = decimal CAGR; 年化波动率小数 = decimal annualized volatility; 最大回撤损失小数 = decimal maximum drawdown loss; 峰值月 = peak month; 谷底月 = trough month.

In [ ]:
results = summarize_all_windows(data, config)
primary = results.loc[results["window_id"] == config["main_window"]].copy()
main_table = primary[[
    "asset_label_zh", "final_wealth_index", "cumulative_simple_return",
    "cagr_month_period_convention", "annualized_volatility_sqrt12_convention",
    "annualized_sharpe_sqrt12_convention", "maximum_drawdown_loss_fraction",
    "maximum_drawdown_peak_month", "maximum_drawdown_trough_month",
]].rename(columns={
    "asset_label_zh": "行业", "final_wealth_index": "期末财富指数",
    "cumulative_simple_return": "累计收益小数", "cagr_month_period_convention": "CAGR 小数",
    "annualized_volatility_sqrt12_convention": "年化波动率小数",
    "annualized_sharpe_sqrt12_convention": "Sharpe",
    "maximum_drawdown_loss_fraction": "最大回撤损失小数",
    "maximum_drawdown_peak_month": "峰值月", "maximum_drawdown_trough_month": "谷底月",
})
display(main_table.round(4))

### Read the Current Main-Window Table

The 2010–2019 window contains 120 continuous constructed months. Wealth starts at 1, so the final index is relative wealth for this constructed path, separate from a security price and the historical report’s HiTec value of 4.6270. CAGR hides the path; volatility measures monthly dispersion; drawdown tracks peak-to-trough pressure; same-month RF Sharpe adds an opportunity-cost comparison. Read the questions these measures answer before ranking them. A single number cannot select a future winner.

## 5. Fixed Segments of the Same Constructed Input

All three 60-month segments below use the Seed101 constructed panel. Fixed slices reveal descriptive window sensitivity in that input, not French market holdouts. Each restarts at initial wealth 1; its drawdown does not inherit the previous segment’s peak.

In [ ]:
segments = results.loc[results["window_id"].isin(["2010_2014", "2015_2019", "2020_2024"]), [
    "window_id", "asset_label_zh", "cagr_month_period_convention",
    "annualized_volatility_sqrt12_convention", "annualized_sharpe_sqrt12_convention",
    "maximum_drawdown_loss_fraction",
]].copy()
segments["CAGR_pct"] = segments.pop("cagr_month_period_convention") * 100
segments["annualized_volatility_pct"] = segments.pop("annualized_volatility_sqrt12_convention") * 100
segments["maximum_drawdown_pct"] = segments.pop("maximum_drawdown_loss_fraction") * 100
display(segments.round(3))

The table above is a Seed101, 180-month constructed method demonstration, separate from the original French report. The accepted Markdown records constructed energy CAGR of approximately +8.008% and Sharpe of +0.623 in 2015–2019, versus −0.262% and −0.073 in 2020–2024. These records are not rerun by this language projection. Interpret signs with their corresponding inputs; the historical report’s negative energy Sharpe is a different result. The preserved asset_label_zh column uses the industry-label translations given above.

## 6. Four Axes in the Constructed Figure

The new figure is written to RUN_DIR/project01_dashboard_CONSTRUCTED.png, still using Seed101 inputs. It covers 2010–2024; shaded 2020–2024 is a descriptive extension. Upper left: month-end labels versus wealth index starting at 1. Upper right: trailing 12-month sample standard deviation times √12, blank for the first 11 months. Lower left: negative drawdown from the running peak. Lower right: industry boxplots of decimal monthly simple returns. Initial wealth is not a price; rolling risk is not a forecast. Preserved plot headings mean 财富指数 = Wealth Index; 滚动波动率 = Rolling Volatility; 峰值回撤 = Drawdown; 月度简单收益分布 = Monthly Simple-Return Distribution. The shared code and its Chinese display labels are unchanged.

In [ ]:
figure_path = make_project_figure(
    data, RUN_DIR / "project01_dashboard_CONSTRUCTED.png", config
)
print(figure_path)
display(Image(filename=str(figure_path)))

## 7. Evidence and Complete Exercises

This Notebook demonstrates metrics, windows and figures for constructed returns. The [original historical report](../research_report/report.md) separately uses French published monthly industry returns and same-month RF, retaining its vintage, historical measures and permissions boundary. Neither provides stock-level OHLCV, historical point-in-time vintages, precise release/ingestion clocks, signals, executions, financing or costs. These descriptive returns and Sharpe values do not establish tradable strategy performance.

Attempt the [six exercises](../exercises.md) before reading the [complete solutions](../solutions.md). Exercise 1 uses +10%/−10%; Exercises 2 and 5 read the original historical report. Do not replace their answers with this constructed table. 